# Multi-Scene Text-to-Video with Sound (CogVideoX-2b + AudioLDM2 + Kokoro)

A longer, more natural-feeling video built from a *sequence of short AI-generated
scenes*, stitched together with crossfade transitions, with sound effects
generated to match each scene's content -- all designed to run on a free
Colab/Kaggle GPU.

**Read this first -- what this notebook can and can't do:**
- No open video model that fits a free GPU generates one continuous minute-long
  shot. [CogVideoX-2b](https://huggingface.co/THUDM/CogVideoX-2b) natively
  produces ~6 seconds (49 frames) per generation. This notebook works around
  that by generating several short scenes and crossfading between them --
  which means visible (if smoothed) transitions, not one unbroken take.
- Sound is generated per scene with [AudioLDM2](https://huggingface.co/cvssp/audioldm2),
  a text-to-*sound-effect* model (birdsong, waves, footsteps, rain -- not
  music), so the audio actually matches what's on screen in each scene,
  instead of one generic soundtrack over the whole thing.
- **License note:** AudioLDM2 is CC-BY-NC-SA-4.0 (non-commercial, share-alike)
  -- different from CogVideoX and Kokoro's Apache 2.0. Keep that in mind if
  this is ever used commercially.
- **Time:** generating ~10 short clips sequentially on a free T4 takes a
  while (each clip is a few minutes). Start with 2-3 scenes first to confirm
  everything works before committing to a full run.

**Run this notebook on:**
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alirezayaned/AI-Video-Generation/blob/main/notebooks/cogvideox_text_to_video.ipynb)
or upload directly to Kaggle (New Notebook -> File -> Import Notebook -> Upload).

## 1. Check your runtime

Make sure you've selected a GPU runtime before running anything below:
- **Colab:** Runtime > Change runtime type > Hardware accelerator > GPU
- **Kaggle:** Notebook Settings (right sidebar) > Accelerator > GPU T4 x2

Free-tier GPUs are not guaranteed to be available on demand, and sessions are
capped (Colab: ~12 hours, Kaggle: ~30 GPU-hours/week, 12 hours/session) -- if
`torch.cuda.is_available()` returns `False` below, you likely need to wait and
retry, or switch platforms.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {vram_gb:.1f} GB")
else:
    print("No GPU detected. Enable a GPU accelerator and restart the runtime before continuing.")


**If you've already hit a CUDA out-of-memory error in this session, restart the kernel/runtime now** (Kaggle: session menu -> restart; Colab: Runtime -> Restart session) **before continuing** -- re-running cells alone does not reliably free GPU memory from a crashed run. The cell below shows how much GPU memory is already in use; if it reports more than a few hundred MB before you've loaded anything, that's leftover from an earlier crash and you need to restart, not just re-run.

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    print(f"GPU memory allocated: {allocated:.2f} GB")
    print(f"GPU memory reserved:  {reserved:.2f} GB")
    if reserved > 0.5:
        print("\nWarning: GPU memory is already in use before loading any model.")
        print("This usually means a previous crash left memory stranded -- restart the kernel/runtime and run from the top.")


## 2. Install dependencies

In [ ]:
!pip install -q --upgrade diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg soundfile


## 3. Load the video model

`THUDM/CogVideoX-2b` downloads in fp16 (~13.8GB of weights, dominated by its
T5-XXL text encoder). Loaded naively that needs ~33GB of VRAM at inference
time -- more than a free T4 has -- so the memory-saving options below are
enabled by default, not optional.

In [ ]:
from diffusers import CogVideoXPipeline

MODEL_ID = "THUDM/CogVideoX-2b"

pipe = CogVideoXPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16)

# Required on free-tier GPUs (T4, ~15GB VRAM). CogVideoX-2b needs ~33GB with no
# offloading at all, and still ~19GB with cpu offload alone -- both exceed a T4.
# cpu offload + VAE tiling together bring it down to ~11GB, which fits with
# headroom to spare.
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()

print("Model loaded (memory-optimized for free-tier GPUs; ~11GB VRAM).")

# If you're running on a larger GPU (A100/L4 40GB+, e.g. Colab Pro) and want
# faster generation instead, replace the three lines above with:
#   pipe = pipe.to("cuda")
# which skips offloading entirely at the cost of needing ~33GB VRAM.


## 4. Describe your video in one prompt

Write **one** sentence or two describing the whole video -- no need to plan
out individual scenes yourself. A small local language model
([Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct),
Apache 2.0) expands it into a shot list behind the scenes: a sequence of
visual scenes with matching ambient-sound descriptions, in the same format
the rest of this notebook already expects.

It runs on **CPU**, not GPU -- this keeps the GPU completely free for
CogVideoX/AudioLDM2 later, and a single one-shot generation like this doesn't
need GPU speed to be practical (expect well under a minute).

The generated scene list prints below so you can review it before the
(much more time-consuming) video generation step. If you want to tweak a
specific scene afterward, just edit the `scenes` list directly in a new
cell, e.g. `scenes[2]["video_prompt"] = "..."` -- it's a plain Python list
at that point.

If you'd rather skip the planner entirely and write out scenes by hand,
see the commented-out example at the bottom of the next cell.

In [ ]:
import gc
import json
import re

from transformers import AutoModelForCausalLM, AutoTokenizer

PLANNER_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

planner_tokenizer = AutoTokenizer.from_pretrained(PLANNER_MODEL_ID)
planner_model = AutoModelForCausalLM.from_pretrained(PLANNER_MODEL_ID, torch_dtype=torch.float32)
# Deliberately NOT moved to "cuda" -- stays on CPU so the GPU is free for
# CogVideoX/AudioLDM2 in later sections.


def parse_scene_json(raw_text):
    text = raw_text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    match = re.search(r"\[.*\]", text, re.DOTALL)
    if match:
        text = match.group(0)
    data = json.loads(text)
    if not isinstance(data, list):
        raise ValueError("Expected a JSON list")
    cleaned = []
    for item in data:
        if isinstance(item, dict) and "video_prompt" in item and "sound_prompt" in item:
            cleaned.append({
                "video_prompt": str(item["video_prompt"]),
                "sound_prompt": str(item["sound_prompt"]),
            })
    if not cleaned:
        raise ValueError("No valid {video_prompt, sound_prompt} objects found")
    return cleaned


def plan_scenes(user_theme, target_scenes=10, max_attempts=3):
    system_prompt = (
        "You are a shot planner for a short AI-generated video montage. Given a single "
        "theme from the user, break it into a sequence of distinct visual scenes that "
        "flow naturally, like a short documentary montage. Respond with ONLY a JSON "
        "array -- no markdown code fences, no commentary before or after. Each element "
        "must be an object with exactly two string keys: \"video_prompt\" (a vivid, "
        "visually specific description for a single 6-second text-to-video shot: "
        "subject, setting, lighting, camera framing) and \"sound_prompt\" (the "
        "real-world ambient sound or sound effects audible in that scene -- not "
        f"music). Produce exactly {target_scenes} scene objects."
    )

    raw = ""
    for attempt in range(max_attempts):
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_theme},
        ]
        text = planner_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = planner_tokenizer([text], return_tensors="pt")

        generated_ids = planner_model.generate(
            **model_inputs,
            max_new_tokens=1500,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
        )
        generated_ids = [
            out[len(inp):] for inp, out in zip(model_inputs.input_ids, generated_ids)
        ]
        raw = planner_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        try:
            return parse_scene_json(raw), raw
        except Exception as e:
            remaining = max_attempts - attempt - 1
            print(f"Attempt {attempt + 1} failed to parse ({e})"
                  + (f"; retrying ({remaining} left)..." if remaining else "."))

    raise RuntimeError(
        "Could not get valid scene JSON after multiple attempts. "
        "Raw output from the last attempt:\n\n" + raw
    )


# ---- Edit this one line to describe your video ----
user_theme = "A peaceful day in nature, from sunrise in the mountains to a starry night, moving through forest, meadow, and shoreline"
TARGET_SCENES = 10  # total length ~= TARGET_SCENES * 6.125s - (TARGET_SCENES - 1) * crossfade_overlap

scenes, _raw_planner_output = plan_scenes(user_theme, target_scenes=TARGET_SCENES)

print(f"Generated {len(scenes)} scenes (requested {TARGET_SCENES}):\n")
for i, s in enumerate(scenes):
    print(f"Scene {i + 1}:")
    print(f"  video: {s['video_prompt']}")
    print(f"  sound: {s['sound_prompt']}")
    print()

del planner_model, planner_tokenizer
gc.collect()

# ---- Alternative: skip the planner and write scenes by hand ----
# scenes = [
#     {
#         "video_prompt": "Sunrise over a misty mountain valley, golden light breaking through clouds, wide shot",
#         "sound_prompt": "birdsong, gentle wind, distant flowing stream",
#     },
#     # ... one dict per scene
# ]


## 5. Generate each clip

This loop generates one silent video per scene. Expect several minutes per
clip on a free T4 -- for 10 scenes, budget 30-60+ minutes total. Progress
prints after each clip so you can see how far along you are.

In [ ]:
import os
import time
from diffusers.utils import export_to_video

os.makedirs("outputs", exist_ok=True)

NUM_FRAMES = 49          # ~6.125s at 8fps -- CogVideoX-2b's native trained length
FPS = 8
NUM_INFERENCE_STEPS = 40 # lower than the usual default of 50 -- a reasonable
                         # quality/time trade-off when generating many clips
GUIDANCE_SCALE = 6.0
CLIP_DURATION = NUM_FRAMES / FPS

def generate_video(prompt, output_path, seed=42):
    generator = torch.Generator(device="cuda").manual_seed(seed)
    frames = pipe(
        prompt=prompt,
        num_frames=NUM_FRAMES,
        num_inference_steps=NUM_INFERENCE_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        generator=generator,
    ).frames[0]
    export_to_video(frames, output_path, fps=FPS)
    return output_path


clip_paths = []
clip_durations = []

for i, scene in enumerate(scenes):
    t0 = time.time()
    path = f"outputs/clip_{i:02d}.mp4"
    generate_video(scene["video_prompt"], path, seed=42 + i)
    clip_paths.append(path)
    clip_durations.append(CLIP_DURATION)
    print(f"[{i + 1}/{len(scenes)}] {path} done in {time.time() - t0:.0f}s")

print("All clips generated.")


## 6. Utility functions: muxing and crossfade concatenation

These are shared by the sections below:
- `mux_audio_video` combines one video with one audio track (used per scene).
- `concatenate_with_crossfade` stitches multiple clips together with a
  crossfade transition instead of a hard cut, for both video and (optionally)
  audio at once.

In [ ]:
import subprocess
import imageio_ffmpeg

FFMPEG_BINARY = imageio_ffmpeg.get_ffmpeg_exe()

def mux_audio_video(video_path, audio_path, output_path):
    """Combine a silent video with an audio track into one file.
    Copies the video stream untouched (fast, no re-encode) and trims to
    the shorter of the two streams so they end together. Forces a fixed
    audio sample rate so clips mux/crossfade consistently later.
    """
    subprocess.run(
        [
            FFMPEG_BINARY, "-y",
            "-i", video_path,
            "-i", audio_path,
            "-c:v", "copy",
            "-c:a", "aac",
            "-ar", "44100",
            "-shortest",
            output_path,
        ],
        check=True, capture_output=True,
    )


def concatenate_with_crossfade(clip_paths, clip_durations, output_path, transition_duration=0.75, with_audio=True):
    """Concatenate clips into one video using a crossfade transition
    (dissolve) between each pair, instead of a hard cut. Crossfades audio
    too when with_audio=True and every clip has an audio stream.
    """
    n = len(clip_paths)
    assert n == len(clip_durations), "clip_paths and clip_durations must match"
    assert n >= 2, "need at least two clips to concatenate"

    inputs = []
    for p in clip_paths:
        inputs += ["-i", p]

    filter_parts = []
    prev_v = "0:v"
    prev_a = "0:a"
    running_duration = clip_durations[0]

    for i in range(1, n):
        offset = running_duration - transition_duration
        v_out = f"v{i}"
        filter_parts.append(
            f"[{prev_v}][{i}:v]xfade=transition=fade:duration={transition_duration}:offset={offset:.3f}[{v_out}]"
        )
        prev_v = v_out

        if with_audio:
            a_out = f"a{i}"
            filter_parts.append(
                f"[{prev_a}][{i}:a]acrossfade=d={transition_duration}[{a_out}]"
            )
            prev_a = a_out

        running_duration = running_duration + clip_durations[i] - transition_duration

    filter_complex = ";".join(filter_parts)

    cmd = [FFMPEG_BINARY, "-y", *inputs, "-filter_complex", filter_complex, "-map", f"[{prev_v}]"]
    if with_audio:
        cmd += ["-map", f"[{prev_a}]", "-c:a", "aac"]
    cmd += ["-c:v", "libx264", output_path]

    subprocess.run(cmd, check=True, capture_output=True)
    return output_path


## 7. Preview the silent montage

Stitches the raw clips together (no audio yet) so you can sanity-check the
sequence and transitions before spending time generating matching sound for
each scene.

In [ ]:
montage_silent_path = "outputs/montage_silent.mp4"
concatenate_with_crossfade(clip_paths, clip_durations, montage_silent_path, transition_duration=0.75, with_audio=False)
print(f"Saved to {montage_silent_path}")


In [ ]:
from IPython.display import Video

Video(montage_silent_path, embed=True, width=480)


## 8. Add realistic ambient sound to each scene

This uses [AudioLDM2](https://huggingface.co/cvssp/audioldm2) (CC-BY-NC-SA-4.0,
non-commercial) -- a text-to-*sound-effect* model, as opposed to a music
generator. It's prompted separately per scene using each scene's
`sound_prompt`, so birdsong plays over the forest shot and waves play over
the beach shot, rather than one generic track over the whole video.

We free the CogVideoX pipeline from GPU memory first -- the free tier doesn't
have room for both models loaded at once.

In [ ]:
import gc

del pipe
gc.collect()
torch.cuda.empty_cache()

from diffusers import AudioLDM2Pipeline, DPMSolverMultistepScheduler

audio_pipe = AudioLDM2Pipeline.from_pretrained("cvssp/audioldm2", torch_dtype=torch.float16)
audio_pipe = audio_pipe.to("cuda")

# AudioLDM2 defaults to 200 DDIM steps; switching schedulers gets equivalent
# quality in ~20 steps (10x faster) -- see huggingface.co/blog/audioldm2
audio_pipe.scheduler = DPMSolverMultistepScheduler.from_config(audio_pipe.scheduler.config)

print("AudioLDM2 loaded.")


In [ ]:
import soundfile as sf

NEGATIVE_AUDIO_PROMPT = "low quality, average quality, noisy, music"

def generate_ambient_sound(prompt, duration_s, output_path, seed=42):
    generator = torch.Generator("cuda").manual_seed(seed)
    audio = audio_pipe(
        prompt,
        negative_prompt=NEGATIVE_AUDIO_PROMPT,
        audio_length_in_s=duration_s,
        num_inference_steps=20,
        generator=generator,
    ).audios[0]
    sf.write(output_path, audio, 16000)
    return output_path


sound_clip_paths = []  # each clip, now with its matching ambient sound muxed in

for i, scene in enumerate(scenes):
    audio_path = f"outputs/clip_{i:02d}_sound.wav"
    generate_ambient_sound(scene["sound_prompt"], clip_durations[i] + 0.5, audio_path, seed=100 + i)

    with_sound_path = f"outputs/clip_{i:02d}_with_sound.mp4"
    mux_audio_video(clip_paths[i], audio_path, with_sound_path)
    os.remove(audio_path)  # keep only the combined output

    sound_clip_paths.append(with_sound_path)
    print(f"[{i + 1}/{len(scenes)}] {with_sound_path} done")

print("All scenes now have matching ambient sound.")


## 9. Preview the final video (crossfaded, with matching sound per scene)

In [ ]:
final_video_path = "outputs/final_video.mp4"
concatenate_with_crossfade(sound_clip_paths, clip_durations, final_video_path, transition_duration=0.75, with_audio=True)
print(f"Saved to {final_video_path}")


In [ ]:
from IPython.display import Video

Video(final_video_path, embed=True, width=480)


## 10. Optional: narrate instead of ambient sound

Narration is a separate final step, and it **replaces** the per-scene
ambient sound rather than layering on top of it (mixing both at once would
need an extra audio-mixing pass -- doable with ffmpeg's `amix` filter if you
want to extend this yourself, but left out here to keep this notebook
manageable).

This uses [Kokoro-82M](https://github.com/hexgrad/kokoro) (Apache 2.0, 82M
params) applied to the *silent* montage from section 7. For a ~54 second
video, that's roughly 100-130 words of natural-paced narration -- longer
text gets cut off at the video's length rather than stretching the video.
See [VOICES.md](https://huggingface.co/hexgrad/Kokoro-82M/blob/main/VOICES.md)
for other available voices.

In [ ]:
!pip install -q kokoro>=0.9.4
!apt-get -qq -y install espeak-ng > /dev/null 2>&1

import gc

for var_name in ("pipe", "audio_pipe"):
    if var_name in globals():
        del globals()[var_name]
gc.collect()
torch.cuda.empty_cache()

from kokoro import KPipeline

tts_pipeline = KPipeline(lang_code='a')  # 'a' = American English
print("Kokoro loaded.")


In [ ]:
import numpy as np

def add_narration(video_path, text, voice="af_heart", output_path=None):
    """Generate speech from text and mux it onto an existing (silent) video."""
    generator = tts_pipeline(text, voice=voice, speed=1, split_pattern=r"\n+")
    audio_chunks = [audio for _, _, audio in generator]
    full_audio = np.concatenate(audio_chunks)

    audio_path = video_path.rsplit(".", 1)[0] + "_narration.wav"
    sf.write(audio_path, full_audio, 24000)

    if output_path is None:
        output_path = video_path.rsplit(".", 1)[0] + "_narrated.mp4"

    mux_audio_video(video_path, audio_path, output_path)
    os.remove(audio_path)
    return output_path


narration_text = """
As the sun rises, mist settles quietly over the valley below.
A trail winds upward through the trees, each step marked by the crunch of fallen leaves.
Life stirs in the open meadow, carefree and unhurried.
By the still waters of the lake, a quiet cabin waits, smoke curling from its chimney.
Rain moves through the forest, feeding the rivers that rush over ancient stone.
As dusk falls, a deer pauses in a clearing, and a campfire begins to glow under the stars.
Far to the west, the ocean catches the last light of the day.
And as night deepens, the mountains stand silent beneath a sky full of stars.
"""

narrated_video = add_narration(montage_silent_path, narration_text)
print(f"Saved to {narrated_video}")


In [ ]:
from IPython.display import Video

Video(narrated_video, embed=True, width=480)


## Notes & next steps

- **Tuning total length:** total duration = `N scenes x 6.125s - (N-1) x transition_duration`.
  Add/remove scenes in section 4, or change `transition_duration` in sections 7/9, to hit a different target length.
- **Tuning speed vs quality:** `NUM_INFERENCE_STEPS` in section 5 (default 40) and
  AudioLDM2's step count in section 8 (default 20) both trade quality for time --
  raise them for a final render, lower them while iterating on scene prompts.
- **License reminder:** CogVideoX-2b and Kokoro are Apache 2.0; AudioLDM2 is
  CC-BY-NC-SA-4.0 (non-commercial). If you swap the narration step in for the
  ambient-sound step, the whole pipeline becomes Apache 2.0 only.
- **Going further:**
  - Layer narration *and* ambient sound together with ffmpeg's `amix` filter instead of choosing one.
  - Swap in [HunyuanVideo-1.5](https://huggingface.co/tencent/HunyuanVideo-1.5)
    via the [Wan2GP](https://github.com/deepbeepmeep/Wan2GP) runner for higher
    quality individual scenes -- still fits in ~6GB VRAM.
  - True single-shot continuous generation beyond ~10s currently needs models
    and GPUs well beyond free-tier (80GB+), e.g. LTX-2.
  - Wrap this pipeline in a [Gradio](https://www.gradio.app/) UI and deploy to
    a free [Hugging Face Space](https://huggingface.co/spaces) for a live demo.
